<a href="https://colab.research.google.com/github/siya-pathak/from-scratch/blob/main/jev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q transformers datasets accelerate scikit-learn pandas

In [2]:
from datasets import load_dataset

train = load_dataset("Heliosoph/Jigsaw-Toxic-Comments", data_files="train.csv.gz", split="train")
print(train[0])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:121: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


README.md:   0%|          | 0.00/8.74k [00:00<?, ?B/s]

train.csv.gz: reconstructing file:   0%|          |  0.00B / 26.7MB            

train.csv.gz: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

{'id': '0000997932d777bf', 'comment_text': "Explanation\nWhy the edits made under my username Hardcore Metallica Fan were reverted? They weren't vandalisms, just closure on some GAs after I voted at New York Dolls FAC. And please don't remove the template from the talk page since I'm retired now.89.205.38.27", 'toxic': 0, 'severe_toxic': 0, 'obscene': 0, 'threat': 0, 'insult': 0, 'identity_hate': 0}


In [3]:
test_text = load_dataset("Heliosoph/Jigsaw-Toxic-Comments", data_files={"test":"test.csv.gz"})

test.csv.gz: reconstructing file:   0%|          |  0.00B / 23.7MB            

test.csv.gz: downloading bytes:           |  0.00B            

Generating test split: 0 examples [00:00, ? examples/s]

In [4]:
test_labels = load_dataset("Heliosoph/Jigsaw-Toxic-Comments", data_files={"test":"test_labels.csv.gz"})

test_labels.csv.gz: reconstructing file:   0%|          |  0.00B / 1.29MB            

test_labels.csv.gz: downloading bytes:           |  0.00B            

Generating test split: 0 examples [00:00, ? examples/s]

Join the test text and test tabels and hugging face dataset had a very complicated way so using pandas instead
later converting df to hugging face dataset

In [5]:
import pandas as pd
from datasets import Dataset

text_df = test_text['test'].to_pandas()
labels_df = test_labels['test'].to_pandas()

test_df = text_df.merge(
    labels_df,
    on="id",
    how="inner",
    validate="one_to_one",
    indicator=True,
)


test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
print(test_dataset)
print(test_dataset[0])
print(test_dataset.features)

Dataset({
    features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate', '_merge'],
    num_rows: 153164
})
{'id': '00001cee341fdb12', 'comment_text': "Yo bitch Ja Rule is more succesful then you'll ever be whats up with you and hating you sad mofuckas...i should bitch slap ur pethedic white faces and get you to kiss my ass you guys sicken me. Ja rule is about pride in da music man. dont diss that shit on him. and nothin is wrong bein like tupac he was a brother too...fuckin white boys get things right next time.,", 'toxic': -1, 'severe_toxic': -1, 'obscene': -1, 'threat': -1, 'insult': -1, 'identity_hate': -1, '_merge': 'both'}
{'id': Value('string'), 'comment_text': Value('string'), 'toxic': Value('int64'), 'severe_toxic': Value('int64'), 'obscene': Value('int64'), 'threat': Value('int64'), 'insult': Value('int64'), 'identity_hate': Value('int64'), '_merge': Value('string')}


Per dataset documentation, labels are jointly missing in test data specifically, all can be -1 for some datapoints. Drop those

In [6]:
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

test_dataset = test_dataset.filter(
    lambda row: all(row[col] != -1 for col in label_cols)
)

print(len(test_dataset))
print({col: sum(test_dataset[col]) for col in label_cols})

Filter:   0%|          | 0/153164 [00:00<?, ? examples/s]

63978
{'toxic': 6090, 'severe_toxic': 367, 'obscene': 3691, 'threat': 211, 'insult': 3427, 'identity_hate': 712}


In [7]:
from collections import Counter
for label in label_cols:
  print(label)
  print(Counter(train[label]))

toxic
Counter({0: 144277, 1: 15294})
severe_toxic
Counter({0: 157976, 1: 1595})
obscene
Counter({0: 151122, 1: 8449})
threat
Counter({0: 159093, 1: 478})
insult
Counter({0: 151694, 1: 7877})
identity_hate
Counter({0: 158166, 1: 1405})


X_train, Y_train, X_val, Y_val

In [8]:
split = train.train_test_split(test_size=0.2, shuffle=True, seed=42)
print(split)

DatasetDict({
    train: Dataset({
        features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 127656
    })
    test: Dataset({
        features: ['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'],
        num_rows: 31915
    })
})


In [9]:
x_train = split["train"]["comment_text"]
y_train = split["train"].select_columns(label_cols)
x_val = split["test"]["comment_text"]
y_val = split["test"].select_columns(label_cols)

In [10]:
print(len(x_train), len(y_train))
print(len(x_val), len(y_val))
print(type(y_train.column_names))

127656 127656
31915 31915
<class 'list'>


Loading the backbone of our JEV which is essentially just going to be our open source model.

From what I understand this is more or less like fine-tuning so our base model needs to stay frozen. Which means **the model needs to be in the eval mode** and we need to verify that no model params are trainable.

In [11]:
from pathlib import Path
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
CACHE_DIR = None
LABELS = tuple(y_train.column_names)
ROOT = Path.cwd()
HEAD_PATH = ROOT / "artifacts" / "jev_head.pt" #this will store the classification head

In [12]:
from transformers import AutoConfig, AutoModel
model = AutoModel.from_pretrained(MODEL_ID, device_map="auto")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [13]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, device_map="auto")

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [14]:
x_train[0]

"Missing Champions \n\nThis article should have ALL the TNA World Heavyweight Champions, from Ken Shamrock. Weird that it doesn't."

In [15]:
inputs = tokenizer(x_train[0], return_tensors="pt")
print(inputs)
input_ids = inputs['input_ids']
attention_mask = inputs['attention_mask']

{'input_ids': tensor([[25080, 28808,  4710,  1986,  4549,  1265,   614, 13097,   279,   350,
          7326,  4337, 28101,  4765, 28808,    11,   504, 14259, 72091, 20521,
            13, 76153,   429,   432,  3171,   944,    13]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1]])}


input_ids: Each number is a token ID representing a token from your comment. Shape (1, 27) means 1 comment with 27 tokens.



attention_mask: A 1 means the token is valid; a 0 means padding that the model should ignore. Here, all 27 tokens are valid.

In [16]:
import torch
from torch import nn

In [18]:
input_device = model.get_input_embeddings().weight.device

input_ids = input_ids.to(input_device)
attention_mask = attention_mask.to(input_device)

model.eval()
for parameter in model.parameters():
    parameter.requires_grad = False

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
        return_dict=True
    )

print(outputs)

BaseModelOutputWithPast(last_hidden_state=tensor([[[-3.7812, -1.5000,  0.3242,  ...,  1.5938,  2.7812, -0.9492],
         [-3.0312,  1.8672,  0.5312,  ...,  2.2656,  1.3672, -0.1729],
         [-0.9023,  1.4141,  2.0156,  ...,  1.9141, -1.3516,  0.2100],
         ...,
         [-1.1719, -0.9336,  4.1875,  ...,  0.1455, -2.2500, -0.7617],
         [-0.5195,  1.2891,  2.0469,  ...,  1.2266,  0.3320, -1.9531],
         [-1.1094, -0.1533,  3.4219,  ...,  0.0216,  1.6953, -0.3164]]],
       device='cuda:0', dtype=torch.bfloat16), past_key_values=None, hidden_states=None, attentions=None)


In [19]:
print(outputs.last_hidden_state.shape)

torch.Size([1, 27, 1536])


1: one comment.



27: 27 tokens in that comment.



1536: each token is represented by a 1536-dimensional contextual vector.


Now we need to pool this
**Because your task is to classify an entire comment, not individual tokens.**
The model gives you 27 vectors, one per token. But your classifier needs a fixed-size representation of the comment to predict labels like toxic, threat, and insult.
Pooling combines those 27 vectors into one 1536-dimensional vector representing the comment. You can then use that representation to predict the six labels.
Think of it as converting token-level information → comment-level information.

**Why do we use the attention mask?** Some comments are shorter than others. When processing comments in batches, shorter ones may be padded with extra tokens.

In [20]:
positions = attention_mask.sum(dim=1) - 1 #we take the last real token, so basically along dimension one (the 2nd dimension), we sum all the ones
#this is the sum of all the real tokens in the sentence (some sentences can have paddings making them have attention mask of 0)
#in our above case all are real tokens so we have positions = 27 - 1(indexing in python starts form 0)
positions = positions.clamp(min=0).long() #in case we have no attention mask, min position has to be 0, converted to long

We are using last vector pooling, assuming the last vector will have the full knowledge of the sentence

In [22]:
hidden = outputs.last_hidden_state #last update after the inputs have gone through the transformer
rows = torch.arange(hidden.size(0), device=hidden.device) #indices are on the same device (CPU or GPU) as the hidden states. #hidden.size(0) is the batch size

In [23]:
print(hidden.shape)

torch.Size([1, 27, 1536])


In [24]:
print(rows)

tensor([0], device='cuda:0')


In [25]:
print(positions)

tensor([26], device='cuda:0')


In [26]:
pooled = hidden[rows, positions] #so for each of the rows, each of the comments, we take the last token of dimension 1536

In [27]:
print(hidden[0, -1, :].shape) #we are using these 1536 and looking at the states one last time before we classify them into one of the multiple label

torch.Size([1536])



(B, 1536) -> (B, 6)


- \(B\): number of comments in a batch.
- 1536: features in each comment’s pooled representation.
- 6: one output score for each toxicity label.

In [28]:
linear = torch.nn.Linear(1536, 6)

In [31]:
linear = linear.to(pooled.device)

logits = linear(pooled.float())
print(logits.shape)  # (B, 6)

torch.Size([1, 6])


In [32]:
#logits and labels shape must match
y = list(y_train[0].values()) #we convert the label to a list to match with logits

BCEWithLogitsLoss calculates a binary classification loss for each of the 6 outputs, then averages the losses by default.

In [34]:
loss_fn = torch.nn.BCEWithLogitsLoss()
labels = torch.tensor(y, dtype=torch.float32).unsqueeze(0) #add the extra 1, dimension to match logits
labels = labels.to(logits.device)
loss = loss_fn(logits, labels)
print(loss.item())

0.1357361376285553


In [35]:
optimizer = torch.optim.AdamW(linear.parameters(), lr=1e-3)

optimizer.zero_grad()
loss.backward()
optimizer.step()

In [64]:
from torch.utils.data import DataLoader, TensorDataset

BATCH_SIZE = 128
EPOCHS = 3
LR = 1e-3
MAX_LENGTH = 256

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [65]:
print(type(x_train))
print(type(x_train[0]))
print(x_train[0])

<class 'datasets.arrow_dataset.Column'>
<class 'str'>
Missing Champions 

This article should have ALL the TNA World Heavyweight Champions, from Ken Shamrock. Weird that it doesn't.


In [66]:
train_tokens = tokenizer(
    list(x_train),
    truncation=True,
    max_length=MAX_LENGTH #We chose 256 as a starting point, not a universal optimum. Some comments may lose information if they're truncated, so we can revisit this choice later.
)

In [67]:
val_tokens = tokenizer(
    list(x_val), truncation=True, max_length=MAX_LENGTH
)

In [68]:
print(type(val_tokens))

<class 'transformers.tokenization_utils_base.BatchEncoding'>


In [69]:
train_ds = Dataset.from_dict(train_tokens)
val_ds = Dataset.from_dict(val_tokens)

In [70]:
train_ds = train_ds.add_column(
    "labels",
    [[y_train[i][c] for c in label_cols]
     for i in range(len(y_train))]
)
val_ds = val_ds.add_column(
    "labels",
    [[y_val[i][c] for c in label_cols]
     for i in range(len(y_val))]
)

In [71]:
from transformers import DataCollatorWithPadding
#helper that makes all comments in a batch the same length by adding padding tokens.
padder = DataCollatorWithPadding(tokenizer=tokenizer)

In [72]:
print(train_ds.column_names)

['input_ids', 'attention_mask', 'labels']


collate all the features(input ids and attention masks) and labels into a single dictionary called batch

In [73]:
def collate_fn(examples): #we send a batch of comments from train_ds
  features = {"input_ids": [], "attention_mask": []}
  label = []
  for example in examples:
    features["input_ids"].append(example["input_ids"])
    features["attention_mask"].append(example["attention_mask"])
    label.append(example['labels'])
  batch = padder(features)
  batch["labels"] = torch.tensor(label, dtype=torch.float32)
  return batch

In [74]:
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE,
    shuffle=True, collate_fn=collate_fn
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_fn
)

In [75]:
#freeeze the base model
model.eval()
for p in model.parameters():
    p.requires_grad = False

In [76]:
print(model.config)

Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 21,
  "model_type": "qwen2",
  

In [77]:
head_device = model.get_input_embeddings().weight.device #just to know where our embedding layer weights are
#classifier will go on the same place as the base model

In [78]:
linear_layer = torch.nn.Linear(model.config.hidden_size, len(label_cols))
linear_layer = linear_layer.to(head_device)

In [79]:
optimizer = torch.optim.AdamW(linear_layer.parameters(), lr=LR)
loss_fn = torch.nn.BCEWithLogitsLoss()

In [ ]:

for epoch in range(EPOCHS):
    linear_layer.train()
    total_loss = 0.0

    # 1. Training
    for batch in train_loader:
        input_device = model.get_input_embeddings().weight.device
        input_ids = batch["input_ids"].to(input_device)
        attention_mask = batch["attention_mask"].to(input_device)
        labels = batch["labels"].to(head_device)

        # Frozen Qwen forward pass
        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
                return_dict=True
            )
            hidden = outputs.last_hidden_state

            mask = attention_mask.to(hidden.device)
            positions = mask.sum(dim=1).sub(1).clamp(min=0).long()
            rows = torch.arange(hidden.size(0), device=hidden.device)
            pooled = hidden[rows, positions]

        # Trainable classification head
        logits = linear_layer(pooled.float().to(head_device))
        loss = loss_fn(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # 2. Validation (after all training batches)
    linear_layer.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            input_device = model.get_input_embeddings().weight.device
            input_ids = batch["input_ids"].to(input_device)
            attention_mask = batch["attention_mask"].to(input_device)
            labels = batch["labels"].to(head_device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
                return_dict=True
            )
            hidden = outputs.last_hidden_state

            mask = attention_mask.to(hidden.device)
            positions = mask.sum(dim=1).sub(1).clamp(min=0).long()
            rows = torch.arange(hidden.size(0), device=hidden.device)
            pooled = hidden[rows, positions]

            logits = linear_layer(pooled.float().to(head_device))
            val_loss += loss_fn(logits, labels).item()

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train loss: {total_loss / len(train_loader):.4f} | "
        f"Val loss: {val_loss / len(val_loader):.4f}"
    )
